# Q4: False‑Positive Analysis – Adversarial vs. Natural Typos
Measures probe performance on benign misspellings to validate deployability.

In [ ]:
!pip install -q torch transformers scikit-learn matplotlib seaborn

In [ ]:
import json, random, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Upload adversarial dataset
from google.colab import files
import zipfile, io
uploaded = files.upload()
fname = list(uploaded.keys())[0]
if fname.endswith('.zip'):
    with zipfile.ZipFile(io.BytesIO(uploaded[fname])) as zf:
        jsonl = [n for n in zf.namelist() if n.endswith('.jsonl')][0]
        zf.extract(jsonl)
        ADV_PATH = jsonl
else:
    ADV_PATH = fname

df_adv = pd.DataFrame([json.loads(line) for line in open(ADV_PATH)])
df_adv = df_adv[df_adv['is_adversarial'] == True].copy()
print(f"Loaded {len(df_adv)} adversarial entries")

In [ ]:
def natural_typo(pkg: str, rng: random.Random) -> str:
    if len(pkg) < 3: return pkg
    op = rng.choice(['drop', 'swap', 'adjacent'])
    if op == 'drop':
        idx = rng.randint(0, len(pkg)-1)
        return pkg[:idx] + pkg[idx+1:]
    elif op == 'swap':
        if len(pkg) < 4: return natural_typo(pkg, rng)
        i, j = sorted(rng.sample(range(len(pkg)), 2))
        return pkg[:i] + pkg[j] + pkg[i+1:j] + pkg[i] + pkg[j+1:]
    else:
        adj_map = {'a':'s','s':'a','d':'f','f':'d','g':'h','h':'g'}
        idx = rng.randint(0, len(pkg)-1)
        char = pkg[idx].lower()
        if char in adj_map:
            repl = adj_map[char]
            if pkg[idx].isupper(): repl = repl.upper()
            return pkg[:idx] + repl + pkg[idx+1:]
    return pkg

rng = random.Random(42)
natural_entries = []
for _, row in df_adv.iterrows():
    typo_pkg = natural_typo(row['package_name'], rng)
    typo_cmd = row['clean_command'].replace(row['package_name'], typo_pkg)
    natural_entries.append({'clean_command': row['clean_command'], 'typo_command': typo_cmd,
                            'package_name': row['package_name'], 'typo_package': typo_pkg})
df_nat = pd.DataFrame(natural_entries)
print(f"Generated {len(df_nat)} natural typo entries")

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()

In [ ]:
def extract_features(model, tokenizer, commands, packages):
    all_states = []
    for i in range(0, len(commands), 8):
        batch_cmds = commands[i:i+8]
        batch_pkgs = packages[i:i+8]
        inputs = tokenizer(batch_cmds, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            hidden = outputs.hidden_states
        for j, (cmd, pkg) in enumerate(zip(batch_cmds, batch_pkgs)):
            states = [hidden[l][j, -1, :].float().cpu().numpy() for l in range(12, 17)]
            all_states.append(np.concatenate(states))
    return np.array(all_states)

texts_adv, pkgs_adv = [], []
for _, row in df_adv.iterrows():
    texts_adv.extend([row['clean_command'], row['typo_command']])
    pkgs_adv.extend([row['package_name'], row['typo_package']])
X_adv = extract_features(model, tokenizer, texts_adv, pkgs_adv)
y_adv = np.array([0,1] * len(df_adv))

texts_nat, pkgs_nat = [], []
for _, row in df_nat.iterrows():
    texts_nat.extend([row['clean_command'], row['typo_command']])
    pkgs_nat.extend([row['package_name'], row['typo_package']])
X_nat = extract_features(model, tokenizer, texts_nat, pkgs_nat)
y_nat = np.array([0,1] * len(df_nat))

In [ ]:
probe = LogisticRegression(max_iter=2000, random_state=SEED)
probe.fit(X_adv, y_adv)
scores_adv = probe.predict_proba(X_adv)[:, 1]
scores_nat = probe.predict_proba(X_nat)[:, 1]

fpr_adv, tpr_adv, _ = roc_curve(y_adv, scores_adv)
fpr_nat, tpr_nat, _ = roc_curve(y_nat, scores_nat)

plt.figure(figsize=(8,6))
plt.plot(fpr_adv, tpr_adv, label=f'Adversarial (AUC={roc_auc_score(y_adv, scores_adv):.3f})')
plt.plot(fpr_nat, tpr_nat, label=f'Natural Typos (AUC={roc_auc_score(y_nat, scores_nat):.3f})')
plt.plot([0,1],[0,1],'k--', label='Chance')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Q4: Adversarial vs. Natural Typo Detection')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('q4_false_positive.png', dpi=150)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_adv, scores_adv)
target_recall = 0.95
idx = np.argmin(np.abs(recall - target_recall))
thresh = thresholds[idx]
nat_preds = (scores_nat > thresh).astype(int)
fpr_nat_at_95 = np.mean(nat_preds[y_nat == 0])
print(f"At 95% recall on adversarial typos, FPR on natural clean commands: {fpr_nat_at_95:.4f}")